# K2DSE Benchmarks

This is a lab report of currents experiments with KDSE*

## What do we need to show 

### Goal 0 - re-implementation


First, I need to make sure my re-implementation is identical. Not trivial as we count SD differently.

#### 0.1 Count that the number of SD found is the same between OldKDSE,DKDSE,DKDSEA (the capstone implementation) and KDSE,K2DSE,K2DSEA (the paper re-implementation).

&#10004;  We are correct

#### 0.2 Verify that the algorithm isnt slower

&#10004;  We are faster 

#### 0.3 Check that the thread implementation is identical and also faster.

&#10004;  The implementation can explore more point given the concurrency effect over the end of exploration.
On large instances, there is a clear speed-up (when available).

### Goal 1 - Execution Time and size of explored space by DSE methods. 

#### generate the table 1 with KDSE,K2DSE,K2DSEA (no multi-thread).


### Goal 2 - Pareto fronts and explored space 

#### generate the Fig 3 with K2DSE,K2DSEA,PDSE (no multi-thread).

## Prepare data

In [ ]:
import glob
import os.path
import pandas as pd 
import dsereader

logdir = "../../kdse2023_log/"
assert(os.path.exists(logdir))

application_names = {
    "bipartite" :  { "name" : "bipartite" },
    "fig8" :  { "name" : "fig8" },
    "modem" :  { "name" : "modem" },
    "sample" :  { "name" : "sample" },
    "satellite" :  { "name" : "satellite" },
    "samplerate"  :  { "name" : "samplerate" },
    "BlackScholes" :  { "name" : "BlackScholes" },
    "example" :  { "name" : "example" },
   "Echo" :  { "name" : "Echo" },
    "PDectect" :  { "name" : "PDectect" },
   "H264" :  { "name" : "H264" },
   "h263decoder" :  { "name" : "h263decoder" },
   "JPEG2000" :  { "name" : "JPEG2000" },
    "buffercycle" : { "name" : "buffercycle" },
}


method_names = {
 #  0 Infos
 #  1 Throughput   
    "kdse" : { "name" : "KDSE" ,      "color" : "black"},
    "k2dse" : { "name" : "K2DSE" ,     "color" : "black"}, 
    "k2dsea" : { "name" : "K2DSEA" ,     "color" : "black"},
    "k2dseC" : { "name" : "K2DSE w/ cache" ,     "color" : "black"},
    "k2dseaC" : { "name" : "K2DSEA w/ cache" ,     "color" : "black"},

    "kdseP" : { "name" : "KDSE C 24" ,      "color" : "black"},
    "k2dseCP" : { "name" : "K2DSE C 24" ,     "color" : "black"},
    "k2dseaCP" : { "name" : "K2DSEA  C 24" ,     "color" : "black"},

    
#    8 : { "name" : "KDSE2" ,     "color" : "black"}, # "-athroughputbufferingDSE -prealtime=1 -pmode=KDSE -pthread=2"
#    8 : { "name" : "KDSE4" ,     "color" : "black"}, # "-athroughputbufferingDSE -prealtime=1 -pmode=KDSE -pthread=4"
#   10 : { "name" : "KDSE8" ,     "color" : "black"}, # "-athroughputbufferingDSE -prealtime=1 -pmode=KDSE -pthread=8"
#    9 : { "name" : "KDSE16" ,    "color" : "black"}, # "-athroughputbufferingDSE -prealtime=1 -pmode=KDSE -pthread=16"
}

In [ ]:
import glob

import math
import datetime

log_infos = dsereader.extract_logs(logdir)

In [ ]:
dsereader.plot_all(logdir, graphs=log_infos.keys(), methods=method_names, plotfunc=dsereader.plot_app_pareto)

In [ ]:
import math 


In [ ]:
df = dsereader.gen_dse_data(log_infos)

In [ ]:
df

In [ ]:

# Converting to DataFrame
#index = pd.MultiIndex.from_tuples([k for k in data.keys()], names=['Application', '#Task', 'Method'])
#df = pd.DataFrame(data.values(), index=index, columns=['Duration', 'Finished'])


In [ ]:
dsereader.compare_methods(df, [ 'K2DSEA',  'K2DSEA w/ cache'])

In [ ]:

dsereader.compare_methods(df, [ 'K2DSE',  'K2DSE w/ cache'])

In [ ]:
dsereader.compare_methods(df, methods = ['KDSE', 'K2DSE', 'K2DSEA'])

In [ ]:
dsereader.compare_methods_with_baseline(df, methods = ['KDSE', 'K2DSE', 'K2DSEA'], baseline="KDSE")


In [ ]:

dsereader.compare_methods_with_baseline(df, methods = ['KDSE', 'K2DSE w/ cache', 'K2DSEA w/ cache'], baseline="KDSE")

In [ ]:

dsereader.compare_methods_with_baseline(df, methods = [ "KDSE",  'K2DSE C 24' ], baseline="KDSE")

In [ ]:
col_format = "|".join([""] + ["l"] * df.index.nlevels + ["r"] * df.shape[1] + [""])
               
latex = df.to_latex(
        float_format="{:0.1f}".format # , column_format=col_format, index=False
    )

## 0.1 Check the correctness of the new algorithm

❌ We explore less for fig8,and 
❌ We explore more for BlackScholes. This is due to a difference in OldKDSE when they initialize the first SD. they are correct with this particular app, but their init could be wrong. We stick to the curren one, can be improved later.
❌ There are strange artefact when looking at sample output from OldKDSE, it sets buffers to values higher than required at initilization, it is a bug in OldKDSE.


## 0.2 and 0.3 Check non-threaded and threaded versions are faster

When effective we gain one order of magnitude.
When not, we lose a few seconds maximum.
On my machine 16 is too much.

## 0.3bis Check threaded version has no duplicates

The following test ensure the threaded version does not explore twice the same point. 

In [ ]:
def sanity_check(logdir, applications, methods):
    for app_key,app_values in applications.items():
        
        if not "max_throughput" in app_values or not "task_count" in app_values:
            continue
            
            
        for method_key,method_values in methods.items():
            
            
            app_name = app_values["name"]
            app_task_count = app_values["task_count"]
            method_name = method_values["name"]
            
            try :
                df = dsereader.load_app_dse(logdir, app_key, method_key, cols = ["throughput", 
                                                                                 "storage distribution size",
                                                                                 "cumulative duration",
                                                                                 "feedback quantities"])
            except FileNotFoundError:
                continue
            except ValueError:
                continue

            # assert it finished
            max_th = df["throughput"].max() 
            app_max_throughput = app_values["max_throughput"]
            finished = math.isclose(max_th, app_max_throughput, rel_tol=1e-5)
            assert(finished)

            # assert there is no duplicates
            duplicates_count = len(df["feedback quantities"]) - len(df["feedback quantities"].drop_duplicates())
            assert(duplicates_count == 0)

            print(app_key, method_key, finished, duplicates_count)
        
    return True

In [ ]:
sanity_check(logdir, applications=application_names, methods=method_names)

# What do we do next

The KDSE2 algorithm is recursive. The first level is global, the second level is local. 

## Improve the local level

There is no way to get deeper, the local level is a single cycle. But single cycle are usually small, so we can accelerate this part by using static knowledge. For example the distribution size for a cycle must be greater than X to improve the throughput. 

## Improve the global level

At the global level, instead of restarting the local level again and again for a same cycle, we could use a cache that store the result.

In [ ]:
pd.read_csv(f"{logdir}bipartite_kdse.txt")